# How people draw and remember: an analysis of chizz

[chizz](https://chizz.party) is a drawing-memory game. A player sees 20 words, one at a time,
with a few seconds each to draw it. The drawings then come back shuffled, and the player has
to say which drawing was which word. Some players guess their own drawings; others guess a
friend's round, sent as a link. The twenty words come from two **silhouette families**, ten
words each. The words in a family draw as the same rough shape (a lion and a sofa), so a
player's own drawings can trick them.

Every visit records the screens it went through, its taps, its drawings stroke by stroke,
and every answer. No identity is attached (see `public/privacy.html`). That gives two
datasets in one: a **memory experiment** with thousands of trials, and a **sketch dataset**
of Turkish words with stroke timings.

**Questions**

1. Where do players stop? (funnel and drop-off)
2. Is the first word remembered best? And the last? (serial position effect)
3. Does more time per word help memory?
4. Does hesitation predict a wrong answer?
5. Do wrong answers stay inside the drawing's silhouette family?
6. Which words are hardest to draw, and which hardest to guess?
7. How do touch and mouse, language, and time of day matter?
8. Do players come back? (retention by cohort)
9. Do shared links bring in new players? (viral loop)
10. Can a model recognise the drawings, and how does it compare with people?

> The data is read from `analysis/data/`, written by `export_data.py`. Before real data
> exists, `fake_data.py` writes a synthetic set with known effects to `analysis/data-fake/`.
> Point `CHIZZ_DATA` there to check that every method finds what was planted.

In [ ]:
import json
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats

import chizz_data as cd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 140)

# Three hues, in fixed order, validated together for colour-blind separation.
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e6e4df"
plt.rcParams.update({
    "figure.figsize": (8, 4), "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left", "lines.linewidth": 2,
})

D = cd.load()
visits, events, rounds, items = D["visits"], D["events"], D["rounds"], D["items"]
print(cd.data_dir())
print(f"{len(visits):,} visits · {len(events):,} events · {len(rounds):,} rounds · {len(items):,} round items")
print("days", visits.day.min(), "to", visits.day.max())

In [ ]:
# The trials used for memory questions: a finished round, guessed by the person who drew it.
own = items[items.role.isin(["solo", "drawer"]) & items.finished_at.notna() & items.correct.notna()].copy()
# Recognition: a finished round guessed by someone else, from a link.
friend = items[(items.role == "guesser") & items.finished_at.notna() & items.correct.notna()].copy()
print(f"{len(own):,} own-memory trials from {own.round_id.nunique():,} rounds")
print(f"{len(friend):,} friend-guess trials from {friend.round_id.nunique():,} rounds")

## 1. Where do players stop?

A visit goes home → countdown → drawing → after drawing → kolay/zor → guessing → result. Count
the share of visits that reach each screen. Also record the screen where each visit ended
(where the tab was closed or left for another app).

In [ ]:
reached = events[events.type == "screen"].groupby("visit_id").screen.agg(set)
reached = reached.reindex(visits.id).apply(lambda s: s if isinstance(s, set) else set())
link = visits.set_index("id").entry.eq("link")
steps = [s for s in cd.FUNNEL if s != "home"]
funnel = pd.DataFrame({
    who: [reached[mask].map(lambda s: step in s).mean() for step in steps]
    for who, mask in [("start page", ~link.values), ("friend's link", link.values)]
}, index=steps)
funnel["start page"] = funnel["start page"] / funnel["start page"].iloc[0]   # of those who pressed start
display((funnel * 100).round(1).rename_axis("reached (%)"))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].bar(range(len(steps)), funnel["start page"] * 100, color=BLUE, width=0.6)
ax[0].set_xticks(range(len(steps)), steps, rotation=30, ha="right")
ax[0].set_title("of those who pressed start, % reaching each screen")
ax[0].set_ylim(0, 105)
ends = visits.last_screen.fillna("?").value_counts(normalize=True).head(8) * 100
ax[1].barh(ends.index[::-1], ends.values[::-1], color=BLUE, height=0.6)
ax[1].set_title("screen the visit ended on, % of visits")
plt.tight_layout()

In [ ]:
# How long a visit lasts, and how much of it is play.
length_min = (visits.last_at - visits.started_at) / 60000
print("visit length, minutes:", length_min.describe(percentiles=[.25, .5, .75, .9]).round(1).to_dict())

## 2. Serial position: is the first word remembered best?

Psychology's **serial position effect** says that in a list, the first items (*primacy*) and
the last ones (*recency*) are recalled better than the middle. chizz gives each word a
position, `idx` 0-19, the order it was drawn in, so the curve can be read straight off the
own-memory trials. Error bars are 95% Wilson intervals.

The trials are not independent: twenty come from each round, and a strong player's round is
strong everywhere. So the test is a logistic regression with standard errors **clustered by
round**. A word's own difficulty is not a confounder here, because a word is equally likely
to land in any position.

In [ ]:
sp = own.groupby("idx").correct.agg(["sum", "count"])
sp["acc"] = sp["sum"] / sp["count"]
lo, hi = cd.wilson(sp["sum"], sp["count"])
fig, ax = plt.subplots()
ax.errorbar(sp.index + 1, sp.acc * 100, yerr=[(sp.acc - lo) * 100, (hi - sp.acc) * 100],
            color=BLUE, marker="o", ms=6, capsize=3, lw=2)
ax.axhline(own.correct.mean() * 100, color=MUTED, lw=1, ls="--")
ax.set_xticks(range(1, 21))
ax.set_xlabel("position the word was drawn in")
ax.set_ylabel("remembered (%)")
ax.set_title("serial position curve, own drawings")
plt.tight_layout()

In [ ]:
own["pos"] = np.select([own.idx <= 2, own.idx >= 17], ["first 3", "last 3"], "middle")
m = smf.logit("correct ~ C(pos, Treatment('middle')) + C(mode)", data=own).fit(
    disp=0, cov_type="cluster", cov_kwds={"groups": pd.factorize(own.round_id)[0]})
odds = pd.DataFrame({"odds ratio": np.exp(m.params), "low": np.exp(m.conf_int()[0]),
                     "high": np.exp(m.conf_int()[1]), "p": m.pvalues}).round(3)
odds

Read the odds ratios against the middle positions. Above 1, with an interval that does not
cross 1, is a real primacy or recency effect. `mode` controls for kolay rounds being easier
than zor rounds.

## 3. Does more time per word help?

In an unlimited round the player picks the seconds per word; the daily is fixed at 3. Because
players choose it, this is **observational**: a careful player may pick more time *and* also
remember better. The model therefore adds the round's mode. Its slope is an association, not
a causal effect.

In [ ]:
un = own[own.kind == "unlimited"]
by = un.groupby("secs").correct.agg(["sum", "count"])
by = by[by["count"] >= 40]
by["acc"] = by["sum"] / by["count"]
lo, hi = cd.wilson(by["sum"], by["count"])
fig, ax = plt.subplots()
ax.errorbar(by.index, by.acc * 100, yerr=[(by.acc - lo) * 100, (hi - by.acc) * 100],
            color=BLUE, marker="o", ms=7, capsize=3)
ax.set_xlabel("seconds per word")
ax.set_ylabel("remembered (%)")
ax.set_title("memory against drawing time, unlimited rounds")
plt.tight_layout()
m = smf.logit("correct ~ secs + C(mode)", data=un).fit(disp=0, cov_type="cluster",
                                                       cov_kwds={"groups": pd.factorize(un.round_id)[0]})
print(f"odds ratio per extra second: {np.exp(m.params['secs']):.2f} "
      f"(95% CI {np.exp(m.conf_int().loc['secs', 0]):.2f}-{np.exp(m.conf_int().loc['secs', 1]):.2f}), p={m.pvalues['secs']:.2g}")
print(by[["count", "acc"]].round(3))

## 4. Does hesitation predict a wrong answer?

Each answer records how many times it was set or changed (`tries`) and when it was finally
given (`answer_ms`, from the grid appearing). Changing an answer is a sign of uncertainty.
Is it a *useful* sign? The test: how well each signal ranks wrong answers above right ones
(ROC AUC, where 0.5 is chance).

In [ ]:
from sklearn.metrics import roc_auc_score
ans = own[own.tries > 0].copy()
ans["wrong"] = 1 - ans.correct
ans["order"] = ans.groupby("round_id").answer_ms.rank()          # 1 = answered first
for col in ["tries", "answer_ms", "order"]:
    print(f"AUC of {col:9} for a wrong answer: {roc_auc_score(ans.wrong, ans[col]):.3f}")
print()
print(ans.groupby(ans.tries.clip(upper=3)).correct.agg(["mean", "count"]).rename(
    index={3: "3+"}, columns={"mean": "accuracy"}).round(3))

If players answer the drawings they are surest of first, the *order* of an answer is a
confidence signal of its own. Compare its AUC with raw time to see whether that holds here.

## 5. Do mistakes stay inside the silhouette family?

The game deals ten words from each of two families, so the shapes collide inside a family. In
kolay, a wrong pick is one of the round's other 19 words. Nine of them share the drawing's
family, so **if mistakes were blind, 9/19 = 47%** would land in the same family. A binomial
test checks whether the observed share is higher.

In [ ]:
wrong = own[(own["mode"] == "pool") & (own.correct == 0) & own.answer_family.notna()]
same = (wrong.family == wrong.answer_family)
k, n = int(same.sum()), len(same)
test = stats.binomtest(k, n, 9 / 19, alternative="greater")
lo, hi = cd.wilson(k, n)
print(f"{k}/{n} = {k/n:.1%} of wrong picks are in the drawing's own family "
      f"(95% CI {lo:.1%}-{hi:.1%}); blind guessing gives 47.4%. p = {test.pvalue:.2g}")

In [ ]:
fam = sorted(set(own.family.dropna()))
cm = pd.crosstab(wrong.family, wrong.answer_family).reindex(index=fam, columns=fam, fill_value=0)
share = cm.div(cm.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(share.values, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(fam)), fam, rotation=60, ha="right")
ax.set_yticks(range(len(fam)), fam)
ax.set_xlabel("family of the word picked")
ax.set_ylabel("family of the drawing")
ax.set_title("where wrong picks go (row = 100%)")
ax.grid(False)
plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout()

In [ ]:
# The single most confused pairs of words.
pairs = Counter(zip(wrong.word, wrong.answer_word))
pd.DataFrame([(a, b, c) for (a, b), c in pairs.most_common(15)], columns=["drawing of", "taken for", "times"])

## 6. Hardest words, two ways

* **hard to remember**: wrong when guessing one's own drawing.
* **hard to draw recognisably**: wrong when a friend guesses it.

A raw percentage over a handful of rounds is noise. So each word's accuracy is shrunk toward
the overall mean with an **empirical-Bayes beta prior**, fitted by the method of moments. A
word seen 4 times moves a lot; one seen 200 times barely moves.

In [ ]:
def shrunk(df, min_n=5):
    g = df.groupby("word").correct.agg(["sum", "count"])
    g = g[g["count"] >= min_n]
    p = g["sum"] / g["count"]
    m, v = p.mean(), p.var()
    strength = max(m * (1 - m) / max(v - (m * (1 - m) / g["count"]).mean(), 1e-4) - 1, 1)   # alpha + beta
    g["acc"] = p
    g["shrunk"] = (g["sum"] + m * strength) / (g["count"] + strength)
    return g.sort_values("shrunk")

mem = shrunk(own)
rec = shrunk(friend, min_n=3) if len(friend) else None
print("hardest to remember");  display(mem.head(12).round(3))
print("easiest to remember");  display(mem.tail(8).round(3))

In [ ]:
if rec is not None and len(rec) > 10:
    both = mem[["shrunk"]].join(rec[["shrunk"]], lsuffix="_memory", rsuffix="_friend", how="inner")
    r, p = stats.spearmanr(both.shrunk_memory, both.shrunk_friend)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(both.shrunk_memory * 100, both.shrunk_friend * 100, s=18, color=BLUE, alpha=0.7,
               edgecolor="white", linewidth=0.8)
    for w in both.assign(d=both.sum(axis=1)).sort_values("d").index[:6]:
        ax.annotate(w, (both.loc[w, "shrunk_memory"] * 100, both.loc[w, "shrunk_friend"] * 100),
                    fontsize=9, color=INK, xytext=(4, 2), textcoords="offset points")
    ax.set_xlabel("remembered by the drawer (%)")
    ax.set_ylabel("recognised by a friend (%)")
    ax.set_title(f"word difficulty, two ways (Spearman ρ = {r:.2f})")
    plt.tight_layout()

## 7. Drawing behaviour: input, language, time of day

What a drawing looks like (strokes, ink, how fast the pen starts) by **touch versus mouse**.
Then whether **ink** makes a drawing easier for a friend to recognise. Language and hour of
day are compared against memory accuracy. These are the checks where *no difference* is also
an answer.

In [ ]:
drawn = items[items.strokes.notna() & (items.strokes > 0)]
display(drawn.groupby("input")[["strokes", "points", "ink", "first_ms", "last_ms"]].median().round(2))
t, p = stats.mannwhitneyu(drawn.loc[drawn.input == "touch", "ink"], drawn.loc[drawn.input == "mouse", "ink"])
print(f"ink, touch vs mouse: Mann-Whitney p = {p:.2g}")

In [ ]:
# A friend's guess against the drawer's ink: the friend's rows carry no drawing, so join it on
# the round code from the drawer's own round.
drawer = items[items.strokes.notna()].merge(rounds[["id", "code"]], left_on="round_id", right_on="id")
fr = friend[["round_id", "idx", "correct"]].merge(rounds[["id", "code"]], left_on="round_id", right_on="id")
rec_ink = fr.merge(drawer[["code", "idx", "ink", "strokes"]].dropna(subset=["code"]).drop_duplicates(["code", "idx"]),
                   on=["code", "idx"], how="inner")
if len(rec_ink) > 50:
    rec_ink["ink_q"] = pd.qcut(rec_ink.ink.rank(method="first"), 5, labels=["least", "2", "3", "4", "most"])
    g = rec_ink.groupby("ink_q", observed=True).correct.agg(["mean", "count"])
    lo, hi = cd.wilson(g["mean"] * g["count"], g["count"])
    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.errorbar(range(5), g["mean"] * 100, yerr=[(g["mean"] - lo) * 100, (hi - g["mean"]) * 100],
                color=BLUE, marker="o", ms=7, capsize=3)
    ax.set_xticks(range(5), g.index)
    ax.set_xlabel("ink in the drawing (quintile)")
    ax.set_ylabel("friend guessed it (%)")
    ax.set_title("more ink, easier to recognise?")
    plt.tight_layout()
else:
    print("not enough friend guesses joined to a drawing yet")

In [ ]:
v = visits.set_index("id")
own_v = own.join(v[["local_hour"]], on="visit_id")
print(own_v.groupby("lang").correct.agg(["mean", "count"]).round(3))
m = smf.logit("correct ~ C(lang) + C(mode)", data=own_v).fit(disp=0, cov_type="cluster",
                                                              cov_kwds={"groups": pd.factorize(own_v.round_id)[0]})
print(f"\nEnglish vs Turkish odds ratio {np.exp(m.params.filter(like='lang').iloc[0]):.2f}, "
      f"p = {m.pvalues.filter(like='lang').iloc[0]:.2g}")

hours = own_v.groupby(pd.cut(own_v.local_hour, [0, 6, 12, 18, 24], right=False,
                             labels=["night 0-6", "morning 6-12", "afternoon 12-18", "evening 18-24"]),
                      observed=True).correct.agg(["mean", "count"])
print(hours.round(3))

## 8. Do players come back?

There is no device id, on purpose. Each visit only says whether the device had played
before, and the first day it played (`first_day`). That is enough for a **cohort table**:
count the visits on day *d* from devices that started on day *c*.

One subtlety: a visit is not a person, since someone can open the game three times a day.
But the **daily round can be played only once per device per day**. So counting daily rounds
instead of visits gives (roughly) unique devices per day, which makes the cohort table a real
retention table.

In [ ]:
dr = rounds[(rounds.kind == "daily") & rounds.drawn_at.notna()].merge(
    visits[["id", "played_before", "first_day"]], left_on="visit_id", right_on="id")
# A device on its first ever visit has no first_day yet: that day is its first.
dr["cohort"] = np.where(dr.played_before == 1, dr.first_day, dr.day).astype(int)
dr["age"] = dr.day - dr.cohort
dr = dr[dr.age >= 0]
size = dr[dr.age == 0].groupby("cohort").size()
table = dr.groupby(["cohort", "age"]).size().unstack(fill_value=0)
ret = table.div(size, axis=0).loc[size[size >= 10].index]
# A day that has not happened yet for a cohort is unknown, not zero.
last_day = dr.day.max()
for c in ret.index:
    ret.loc[c, [a for a in ret.columns if c + a > last_day]] = np.nan
later = ret.drop(columns=0)                                 # day 0 is the cohort itself, 100%
weekly = later.T.groupby((later.columns - 1) // 7).mean().T  # days 1-7, 8-14, ...
fig, ax = plt.subplots(figsize=(9, 5))
cols = [c for c in weekly.columns if c <= 4]
cmap = plt.get_cmap("Blues").copy()
cmap.set_bad("white")
im = ax.imshow(np.ma.masked_invalid(weekly[cols].values * 100), cmap=cmap, aspect="auto", vmin=0)
ax.set_xticks(range(len(cols)), [f"days {7 * int(c) + 1}-{7 * int(c) + 7}" for c in cols])
ax.set_yticks(range(len(weekly)), [f"day {c}" for c in weekly.index])
ax.set_xlabel("days since first daily (blank: not reached yet)")
ax.set_ylabel("cohort (first daily)")
ax.set_title("share of a cohort playing the daily on an average day (%)")
ax.grid(False)
plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout()

curve = ret.mean()
print("average day-1 / day-7 / day-14 retention:",
      " / ".join(f"{curve.get(d, np.nan):.1%}" for d in (1, 7, 14)))

In [ ]:
# Streaks, as seen at the start of a visit: how many dailies in the last 14 days.
visits.loc[visits.played_before == 1, "days_played"].value_counts().sort_index().plot.bar(color=BLUE, width=0.7)
plt.title("dailies played in the last 14 days, returning visits")
plt.xlabel("days")
plt.ylabel("visits")
plt.tight_layout()

## 9. Do shared links bring new players?

A shared round carries a code. A visit that arrives by a link records the code it came with
(`via_code`), and the round that visit then plays can be shared again. That gives the loop:

**shared → link opened → friend finished guessing → friend drew their own → friend shared**

The **k-factor** is the number of new players each shared round brings in. Above 1 the game
grows on its own. Below 1, links help, but the game still needs new players from elsewhere.

In [ ]:
shares = events[events.type == "share"]
link_visits = visits[visits.entry == "link"]
lr = rounds.merge(link_visits[["id", "via_code"]], left_on="visit_id", right_on="id", suffixes=("", "_v"))
shared_codes = set(shares.code.dropna())
guessed = lr[(lr.role == "guesser") & lr.finished_at.notna()].visit_id.unique()
drew = lr[(lr.role == "solo") & lr.drawn_at.notna()].visit_id.unique()
reshared = shares[shares.visit_id.isin(link_visits.id)].visit_id.unique()

steps = pd.Series({
    "rounds shared": len(shared_codes),
    "link visits": len(link_visits),
    "…finished guessing": len(guessed),
    "…drew their own": len(drew),
    "…shared again": len(reshared),
})
display(steps.to_frame("count"))
print(f"links opened per shared round: {len(link_visits) / max(len(shared_codes), 1):.2f}")
print(f"k-factor (new players who went on to play a round, per shared round): "
      f"{len(set(guessed) | set(drew)) / max(len(shared_codes), 1):.2f}")

In [ ]:
# How deep do chains go? Follow code -> link visit -> the code that visit's own round got.
code_of_visit = rounds.dropna(subset=["code"]).groupby("visit_id").code.agg(set)
children = {}
for v in link_visits.itertuples():
    for c in code_of_visit.get(v.id, ()):
        if c in shared_codes and c != v.via_code:
            children.setdefault(v.via_code, set()).add(c)
def depth(c, seen=()):
    return 1 + max((depth(k, seen + (c,)) for k in children.get(c, ()) if k not in seen), default=0)
roots = shared_codes - {k for ks in children.values() for k in ks}
print("chain depth (1 = shared, nobody passed it on):")
print(pd.Series([depth(c) for c in roots]).value_counts().sort_index())

## 10. Can a model recognise the drawings?

Each drawing is turned into a 28×28 image, scaled to fill the frame as in Google's Quick, Draw!
data, so size and position don't matter. Two models are trained on the words with enough
examples, and the split keeps **whole rounds apart**: no round appears in both train and test.

* logistic regression on the pixels, as the baseline
* a small neural network (MLP)

Each word is a class. People only had to tell 20 words apart. The model faces every word
with enough data at once, which is a much harder task. So **top-1 / top-5 accuracy** is
compared with chance, and with a fairer human-like test: picking the right word from the 20
words of the drawing's own round.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import top_k_accuracy_score

MIN_PER_WORD = 25
MAX_WORDS = 60          # the most drawn words; keeps training to a minute or two
dr_items = items[items.drawing.notna() & (items.strokes > 0)]
counts = dr_items.word.value_counts()
keep = dr_items[dr_items.word.isin(counts[counts >= MIN_PER_WORD].index[:MAX_WORDS])].reset_index(drop=True)
print(f"{len(keep):,} drawings of {keep.word.nunique()} words with ≥{MIN_PER_WORD} drawings each")

if keep.word.nunique() >= 5:
    X = np.stack([cd.rasterize(d).ravel() for d in keep.drawing])
    y = keep.word.values
    tr, te = next(GroupShuffleSplit(test_size=0.2, random_state=0).split(X, y, keep.round_id))
    models = {
        "logistic regression": LogisticRegression(max_iter=300, C=0.5),
        "MLP (256)": MLPClassifier(hidden_layer_sizes=(256,), alpha=1e-3, max_iter=60, early_stopping=True, random_state=0),
    }
    results = {}
    for name, mdl in models.items():
        mdl.fit(X[tr], y[tr])
        P = mdl.predict_proba(X[te])
        results[name] = {"top-1": top_k_accuracy_score(y[te], P, k=1, labels=mdl.classes_),
                         "top-5": top_k_accuracy_score(y[te], P, k=5, labels=mdl.classes_)}
    results["chance"] = {"top-1": 1 / len(set(y)), "top-5": 5 / len(set(y))}
    display(pd.DataFrame(results).T.round(3))

In [ ]:
# The human-like test: for each test drawing, choose among the 20 words of its own round only.
if keep.word.nunique() >= 5:
    mdl = models["MLP (256)"]
    P = pd.DataFrame(mdl.predict_proba(X[te]), columns=mdl.classes_)
    words_of = rounds.set_index("id").words
    hits = []
    for i, row in enumerate(keep.iloc[te].itertuples()):
        cand = [w for w in words_of[row.round_id] if w in P.columns]
        if row.word in cand and len(cand) > 1:
            hits.append(P.loc[i, cand].idxmax() == row.word)
    human = own.correct.mean()
    print(f"model, picking among its round's words it knows: {np.mean(hits):.1%} (n={len(hits)})")
    print(f"people, remembering their own drawings:           {human:.1%}")
    print(f"people, recognising a friend's drawings:          {friend.correct.mean():.1%}")

In [ ]:
# The drawings in the Quick, Draw! format, for anyone who wants to train on them elsewhere.
cd.quickdraw_ndjson(items, cd.data_dir() / "chizz_drawings.ndjson")
print("written", cd.data_dir() / "chizz_drawings.ndjson")

# A look at what the data is: a few drawings of the most drawn word.
w = counts.index[0]
sample = dr_items[dr_items.word == w].head(8)
fig, axs = plt.subplots(1, len(sample), figsize=(1.6 * len(sample), 1.9))
for ax, d in zip(np.atleast_1d(axs), sample.drawing):
    for xs, ys, _ in cd.strokes(d):
        ax.plot(xs, ys, color=INK, lw=1.6)
    ax.set_xlim(0, 255); ax.set_ylim(255, 0); ax.set_aspect("equal"); ax.axis("off")
fig.suptitle(f"“{w}”, as eight players drew it", x=0.01, ha="left", fontweight="bold")
plt.tight_layout()

## What to take from it

Fill this in from the real numbers once they are in. A good write-up states each finding
with its size and its uncertainty, and says what it changes. For example: *"wrong picks fall
inside the silhouette family 2× more often than blind chance (p < 0.001). The family design
does what it is meant to."* Then it names the limits:

* **No people, only visits.** Retention rests on the once-a-day daily, and a device that
  clears its storage starts over as a new cohort.
* **Self-selected settings.** The seconds per word are chosen by the player, so section 3 is
  an association.
* **Who plays.** Players come from friends' links. They are not a random sample of anyone.